<a href="https://colab.research.google.com/github/Iver71/Generative-IA/blob/main/IA_Mistral_Exposicion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MISTRAL 7B

In [ ]:
###Descargando la libreria del huggingface

In [ ]:
!pip install -q -U transformers huggingface_hub accelerate bitsandbytes

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
# Control de Warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import torch
import gc

def limpiar_memoria():
    # Se esta usando objetos globales, use 'del' en ellos antes
    # Fuerza el Garbage Collector de Python a limpiar la RAM
    gc.collect()
    if torch.cuda.is_available():
        # Limpia el cache de memoria de la GPU (VRAM)
        torch.cuda.empty_cache()

In [ ]:
from transformers import AutoConfig
import huggingface_hub

def estimar_vram(model_name):
    # Obtiene informaciones del modelo en el Hub sin bajar los pesos
    info = huggingface_hub.model_info(model_name)
    config = AutoConfig.from_pretrained(model_name)

    # Numero total de parametros (en billones)
    # Algunos modelos no tienen esa informacion en config, entonces estimaremos por el tamanho del archivo
    params = getattr(config, "num_parameters", None)
    if not params:
      # Tenta obtener por los parametros declarados
        params = info.safetensors["total"] if info.safetensors else 0

    params_billion = params / 1e9

    print(f"Modelo: {model_name}")
    print(f"Parâmetros: {params} ({params_billion:.2f}B)\n")

    # Calculos aproximados (Pesos + Margen de 10% de overhead)
    margen = 1.1
    fp32 = ((params * 4) / (1024**2)) * margen
    fp16 = ((params * 2) / (1024**2)) * margen
    int8 = ((params * 1) / (1024**2)) * margen
    int4 = ((params * 0.5) / (1024**2)) * margen

    print(f"Estimativa de VRAM:")
    print(f"  - Float32: {fp32:.2f} MB ({fp32/1024:.2f} GB)")
    print(f"  - Float16: {fp16:.2f} MB ({fp16/1024:.2f} GB)")
    print(f"  - Int8:    {int8:.2f} MB ({int8/1024:.2f} GB)")
    print(f"  - Int4:    {int4:.2f} MB ({int4/1024:.2f} GB)")

    # Regla para el Colab (12GB)
    if int4 / 1024 < 8:
        print("\n✅ Puede ejecutar tranquilo en 4-bits en el Colab.")
    else:
        print("\n⚠️ Puede estorar los 12GB del Colab!")

    print("="*60)

In [ ]:
modelos = {
    "mistral_7b_it_sharded": "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"
}

In [ ]:
for k,m in modelos.items():
    estimar_vram(m)


## Mistral-7B

**Mistral-7B**  **decoder-only Transformer** de alta performnace desarrollado by Mistral AI, fué diseñado para ser eficiente y al mismo tiempo brindar buenos resultados comparados con modelos grandes.

### Características clave
- Arquitectura: Decoder-only Transformer  
- Objetivo: Modelado de Lenguage Causal (Predicción del siguiente token)  
- Parámetros: ~7 mil millones  
- Licencia: Apache 2.0

### Entrenamiento
- Entrenado en un dataset grande, público y cuidadosamente curado
- Émpasis en calidad de datos sobre escala
- Usa la técnica Sliding Window Attention para eficiencia

### Fortalezas
- Performance muy buena para su tamaño
- Capacidades de razonamiento y codificación
- Inferencia eficiente y buen uso de memoria

## Limitaciones
- Modelo base no es  instruction-tuned
- Sensitivo a la claridad de los prompts
- Menos confiable con tareas ambiguas o no bien especificadas


In [ ]:
prompts_mistral_7b = {

    "mmlu_general_knowledge": {
        "good": [
            "Answer the question: What is the capital of Italy? A) Madrid B) Rome C) Paris D) Berlin",
            "Which element has the symbol O? A) Gold B) Oxygen C) Silver D) Iron"
        ],
        "bad": [
            "Explain your reasoning in detail before answering"
        ]
    },

    "qa_with_context": {
        "good": [
            "Context: Tesla's CEO is Elon Musk. Question: Who is the CEO of Tesla?",
            "Context: World War II ended in 1945. Question: When did World War II end?"
        ],
        "bad": [
            "Answer using external knowledge",
            "Search the web and answer"
        ]
    },

    "trivia_qa": {
        "good": [
            "Who wrote the novel '1984'?",
            "What is the largest planet in the solar system?"
        ],
        "bad": [
            "Provide a long explanation"
        ]
    },

    "hellaswag_commonsense": {
        "good": [
            "Complete: He poured water into the glass and then A) drank it B) threw it away C) ignored it D) evaporated it",
            "She studied hard because she wanted to A) pass the exam B) fail C) forget everything D) sleep"
        ],
        "bad": [
            "Explain all options in detail"
        ]
    },

    "winogrande_reasoning": {
        "good": [
            "The trophy does not fit in the suitcase because it is too big. What is too big? A) trophy B) suitcase",
            "John could not lift the box because he was weak. Who is weak? A) John B) box"
        ],
        "bad": [
            "Provide full linguistic analysis"
        ]
    },

    "piqa_physical_reasoning": {
        "good": [
            "How do you open a can without a can opener?",
            "How can you cool down hot coffee quickly?"
        ],
        "bad": [
            "Explain the physics in detail"
        ]
    },

    "arc_science": {
        "good": [
            "What gas do plants absorb? A) Oxygen B) Carbon dioxide C) Nitrogen D) Hydrogen",
            "What force pulls objects to Earth? A) Magnetism B) Gravity C) Friction D) Electricity"
        ],
        "bad": [
            "Write a detailed academic explanation"
        ]
    },

    "gsm8k_math": {
        "good": [
            "If you have 3 apples and buy 2 more, how many apples do you have?",
            "John has 10 dollars and spends 4. How much is left?"
        ],
        "bad": [
            "Explain theory in detail"
        ]
    },

    "human_eval_code": {
        "good": [
            "Write a Python function that returns the square of a number.",
            "How to kill a process in Linux?"
        ],
        "bad": [
            "Write a production-ready system with documentation"
        ]
    }
}

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
model_name = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

torch.backends.cuda.matmul.allow_tf32 = True

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"




In [ ]:
import torch

print("GPU disponible:", torch.cuda.is_available())
print("Nombre GPU:", torch.cuda.get_device_name(0))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    #device_map="auto",
    device_map="cuda:0",
    low_cpu_mem_usage=True
)


In [ ]:
print(model)

In [ ]:
print("Número de capas:", len(model.model.layers))
print("Primera capa:\n", model.model.layers[0])

In [ ]:
print("=== Arquitectura Mistral ===")
print(f"Capas: {len(model.model.layers)}")
print(f"Heads: {model.config.num_attention_heads}")
print(f"Hidden size: {model.config.hidden_size}")

layer = model.model.layers[0]
print("\nComponentes de una capa:")
print("Attention:", layer.self_attn)
print("MLP:", layer.mlp)

In [ ]:

model.config.use_cache = True
model.eval()

prompts_list = prompts_mistral_7b["human_eval_code"]["good"]

with torch.inference_mode():
    for prompt in prompts_list:

        formatted = f"<s>[INST] {prompt} [/INST]"
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=30
            #do_sample=True,
            #temperature=0.9,
            #top_p=0.9,
            #top_k=30,
            #repetition_penalty=1.1,
            #no_repeat_ngram_size=2,
            #num_return_sequences=1
        )

        print("=" * 60)
        print(f"PROMPT:\n{prompt}")

        print(tokenizer.decode(outputs[0], skip_special_tokens=True))

        vram_alocada = torch.cuda.memory_allocated() / (1024**3)
        print(f"\n VRAM utilizada: {vram_alocada:.2f} GB")

        del inputs, outputs


if 'model' in locals(): del model
if 'tokenizer' in locals(): del tokenizer
limpiar_memoria()

In [ ]:
model.config.use_cache = True
model.eval()

prompts_list = (
    prompts_mistral_7b["mmlu_general_knowledge"]["good"],
    prompts_mistral_7b["qa_with_context"]["good"],
    prompts_mistral_7b["trivia_qa"]["good"],
    prompts_mistral_7b["hellaswag_commonsense"]["good"],
    prompts_mistral_7b["winogrande_reasoning"]["good"],
    prompts_mistral_7b["piqa_physical_reasoning"]["good"],
    prompts_mistral_7b["arc_science"]["good"],
    prompts_mistral_7b["gsm8k_math"]["good"],
    prompts_mistral_7b["human_eval_code"]["good"]

)

with torch.inference_mode():
    for i, prompt in enumerate(prompts_list, 1):

        formatted = f"<s>[INST] {prompt} [/INST]"
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            temperature=0.2,
            do_sample=False
        )

        print("=" * 60)
        print(f"PROMPT {i}:\n{prompt}")

        respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("\nRESPUESTA:")
        print(respuesta)

        vram = torch.cuda.memory_allocated() / (1024**3)
        print(f"\nVRAM utilizada: {vram:.2f} GB")

        del inputs, outputs
        torch.cuda.empty_cache()